In [435]:
import pickle
import json
import numpy as np
import cv2
from matplotlib import pyplot as plt
from scipy.ndimage import uniform_filter
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

In [436]:
def snr(clean, noisy):
    noise = clean.astype(float) - noisy.astype(float)
    power_signal = np.sum(clean.astype(float)**2)
    power_noise  = np.sum(noise**2)
    return 10 * np.log10(power_signal / power_noise)

def median_filter(img, size=7):
    """
    img : float32 entre 0 et 1
    size : taille du voisinage (3,5,7…)
    """
    img_norm = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)
    img_uint8 = img_norm.astype(np.uint8)
    denoised = cv2.medianBlur(img_uint8, size)
    return denoised.astype(np.float32) / 255.0

    

In [437]:

with open('../../../ETL/data/pickles/processed.pkl', 'rb') as file:
	data = pickle.load(file)



with open('results_medianFilter.json', 'r') as file :
    results = json.load(file)

In [438]:
with open('../../../ETL/data/L.txt', 'r') as file:
    l = file.readline()

L = int(l)

In [439]:
L

2

In [440]:
r = []
for i in range(12):
    r.append({})
    for size in [3, 5, 7, 9, 11] : 
        caseValue = 'size : ' + str(size)
        r[-1][caseValue] = {}
        img = data['set12']['clean'][i]
        noisy_img = data['set12']['noisy'][i]
        r[-1][caseValue]['snr'] = snr(img, noisy_img)

        filtered_img = median_filter(noisy_img, size=size)
        
        r[-1][caseValue]['psnr'] = psnr(img, filtered_img, data_range=1.0)
        r[-1][caseValue]['ssim'] = ssim(img, filtered_img, data_range=1.0)


In [441]:
rs = {i : 
    {
        'snr': np.mean([j[i]['snr'] for j in r]),
        'psnr': np.mean([j[i]['psnr'] for j in r]),
        'ssim': np.mean([j[i]['ssim'] for j in r]),
        'real var' : 1/L
    } for i in r[0].keys()} 

In [442]:
#rs

In [443]:
results.keys()

dict_keys(['1', '2', '3', '5', '7', '10', '13', '16', '20', '25', '30', '40', '50', '70', '100', '4'])

In [444]:
results[str(L)] = rs

In [445]:
results.keys()

dict_keys(['1', '2', '3', '5', '7', '10', '13', '16', '20', '25', '30', '40', '50', '70', '100', '4'])

In [446]:
with open('results_medianFilter.json', 'w') as file:
    json.dump(results, file)